In [ ]:
# top 300 QTSA test with RAG integration

import torch, os, re
from openai import OpenAI
import json
import time
import textwrap
import numpy as np
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer
from FlagEmbedding import BGEM3FlagModel, FlagReranker
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

os.environ["CUDA_VISIBLE_DEVICES"] = "2"
BASE_DIR = "path to your project directory"  # Change this to your project directory
INPUT_FILE = "path to your input file"  # Change this to your input file
OUTPUT_FILE = "path to your output file"  # Change this to your output file
MODEL_NAME = "gpt-4o"
client = OpenAI(api_key="your api key")  # Change this to your OpenAI API key
        
# ===========================main()========================
def main():

    jsonl_path = INPUT_FILE
    output_path = OUTPUT_FILE
    # context_output_path = CONTEXT_FILE
    
    total_start_time = time.time()
    
    results = []
    max_items = 1000
    
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if not line.strip():
                continue
                
            if i >= max_items:
                break
            
            try:
                data = json.loads(line.strip())
                question = data.get("question", "")
                correct_answer = data.get("answer", "")
                id_num = data.get("id", i + 1)
                
                print(f"\n[{i+1}/{max_items}] Processing ID {id_num}: {question[:60]}...")

                # ====== set up prompt ======
                system_prompt = """You are a Radio Frequency expert. Please answer questions using concise and professional language, providing the core content directly without excessive explanation."""
                
                user_prompt = f"""Based on the following reference information, please answer the question:

Question: {question}

Please provide a concise and accurate answer based on the reference information above. If the information is not sufficient, use your professional knowledge to answer."""
                
                prompt = f"""<|im_start|>system
{system_prompt}
<|im_end|>
<|im_start|>user
{user_prompt}
<|im_end|>
<|im_start|>assistant
"""
                
                start_time = time.time()

                start_time = time.time()

                response = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                    ],
                    temperature=0,
                    max_tokens=1000,
                )

                inference_time = time.time() - start_time
                full_response = response.choices[0].message.content
                input_tokens_count = response.usage.prompt_tokens
                output_tokens_count = response.usage.completion_tokens
                print(f"  Output tokens: {output_tokens_count}")
                print(f"  Inference time: {inference_time:.2f}s")
                
                # inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
                # input_tokens_count = inputs.input_ids.shape[1]
                # print(f"  Input tokens: {input_tokens_count}")
                
                # with torch.no_grad():
                #     outputs = model.generate(
                #         **inputs,
                #         max_new_tokens=16000,  
                #         repetition_penalty=1.2,
                #         do_sample=False,
                #         pad_token_id=tokenizer.eos_token_id,
                #         eos_token_id=tokenizer.eos_token_id,
                #     )
                
                # output_tokens_count = outputs[0].shape[0] - inputs.input_ids.shape[1]
                # full_response = tokenizer.decode(
                #     outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
                # )
                
                # result = {
                #     "id": id_num,
                #     "question": question,
                #     "rag_context": context,
                #     "model_output": full_response,
                #     "correct_answer": correct_answer,
                #     "rag_time": rag_time,
                #     "inference_time": inference_time,
                #     "total_time": rag_time + inference_time,
                #     "input_tokens": input_tokens_count,
                #     "output_tokens": output_tokens_count
                # }
                result = {
                    "id": id_num,
                    "question": question,
                    "model_output": full_response,
                    "correct_answer": correct_answer,
                    "inference_time": inference_time,
                    "input_tokens": input_tokens_count,
                    "output_tokens": output_tokens_count, 
                    "model_name": MODEL_NAME,
                }
                results.append(result)

                print(f"  Model response: {full_response[:80]}...")
                
            except json.JSONDecodeError as e:
                print(f"Line {i+1} JSON decode error: {e}")
            except Exception as e:
                print(f"Error processing line {i+1}: {e}")
                import traceback
                traceback.print_exc()

    total_end_time = time.time()
    total_processing_time = total_end_time - total_start_time
    
    if results:
        avg_time_per_question = total_processing_time / len(results)
        avg_inference_time = sum(r["inference_time"] for r in results) / len(results)
        avg_input_tokens = sum(r["input_tokens"] for r in results) / len(results)
        avg_output_tokens = sum(r["output_tokens"] for r in results) / len(results)
        
        print(f"\n{'='*60}")
        print("SUMMARY")
        print(f"{'='*60}")
        print(f"Total questions processed: {len(results)}")
        print(f"Total processing time: {total_processing_time:.2f}s")
        print(f"Average time per question: {total_processing_time/len(results):.2f}s")
        print(f"Average inference time: {avg_inference_time:.2f}s")
        print(f"Average input tokens: {avg_input_tokens:.1f}")
        print(f"Average output tokens: {avg_output_tokens:.1f}")
        print(f"{'='*60}")

    with open(output_path, "w", encoding="utf-8") as f:
        for result in results:
            f.write(json.dumps(result, ensure_ascii=False) + "\n")
    
    print(f"\nComplete! Results saved to: {output_path}")
    # print(f"RAG contexts saved to: {context_output_path}")
    print(f"Processed {len(results)} questions")

if __name__ == "__main__":
    main()